In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/models/tanmay240/mlp-scratch/pytorch/default/1/mlp_scratch/tfidf_vectorizer.pkl
/kaggle/input/models/tanmay240/mlp-scratch/pytorch/default/1/mlp_scratch/model.pt
/kaggle/input/models/tanmay240/bert-base-uncased/pytorch/default/1/bert-base-uncased/tokenizer.json
/kaggle/input/models/tanmay240/bert-base-uncased/pytorch/default/1/bert-base-uncased/tokenizer_config.json
/kaggle/input/models/tanmay240/bert-base-uncased/pytorch/default/1/bert-base-uncased/model.pt
/kaggle/input/models/tanmay240/roberta-base/pytorch/default/1/roberta-base/tokenizer.json
/kaggle/input/models/tanmay240/roberta-base/pytorch/default/1/roberta-base/tokenizer_config.json
/kaggle/input/models/tanmay240/roberta-base/pytorch/default/1/roberta-base/model.pt
/kaggle/input/models/tanmay240/electra-base-discriminator/pytorch/default/1/electra-base-discriminator/tokenizer.json
/kaggle/input/models/tanmay240/electra-base-discriminator/pytorch/default/1/electra-base-discriminator/tokenizer_config.json
/kaggle/i

In [2]:
import os
import gc
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel

# setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
sample = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')
option_letters = ['A', 'B', 'C', 'D', 'E']

# custom architecture
class SmartMCQPretrained(nn.Module):
    def __init__(self, model_name):
        super(SmartMCQPretrained, self).__init__()
        self.transformer = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.transformer.config.hidden_size, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, input_ids, attention_mask):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :] 
        x = self.dropout(cls_output)
        x = self.classifier(x)
        return self.sigmoid(x).squeeze(-1)

# inference pipeline
def extract_model_probabilities(model_dir, hf_base_name):
    print(f"\nLoading Tokenizer and Model from: {model_dir}")
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = SmartMCQPretrained(hf_base_name).to(device)
    
    # load trained weights
    model.load_state_dict(torch.load(os.path.join(model_dir, "model.pt"), map_location=device))
    model.eval()
    
    all_scores = []
    
    with torch.no_grad():
        for index, row in tqdm(test_df.iterrows(), total=len(test_df), desc=f"Predicting {hf_base_name}"):
            prompt = str(row['prompt'])
            question_scores = []
            
            for letter in option_letters:
                option_text = str(row[letter])
                combined_text = f"{prompt} {option_text}"
                
                encoding = tokenizer(
                    combined_text,
                    truncation=True,
                    padding='max_length',
                    max_length=128,
                    return_tensors='pt'
                ).to(device)
                
                prob = model(encoding['input_ids'], encoding['attention_mask']).item()
                question_scores.append(prob)
                
            all_scores.append(question_scores)
            
    # clear vram for next model
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    
    return np.array(all_scores)

# ensemble submission
model_paths = {
    "google/electra-base-discriminator": "/kaggle/input/models/tanmay240/electra-base-discriminator/pytorch/default/1/electra-base-discriminator",
    "google-bert/bert-base-uncased": "/kaggle/input/models/tanmay240/bert-base-uncased/pytorch/default/1/bert-base-uncased",
    "FacebookAI/roberta-base": "/kaggle/input/models/tanmay240/roberta-base/pytorch/default/1/roberta-base"
}

# model probabilities
elec_probs = extract_model_probabilities(model_paths["google/electra-base-discriminator"], "google/electra-base-discriminator")
bert_probs = extract_model_probabilities(model_paths["google-bert/bert-base-uncased"], "google-bert/bert-base-uncased")
rob_probs = extract_model_probabilities(model_paths["FacebookAI/roberta-base"], "FacebookAI/roberta-base")

# hyperparameter weights
w_rob = 0.95
w_elec = 0.04
w_bert = 0.01

print("\nBlending probabilities and ranking answers...")
blended_probs = (w_elec * elec_probs) + (w_bert * bert_probs) + (w_rob * rob_probs)

for i in range(len(test_df)):
    ranked_indices = np.argsort(blended_probs[i])[::-1]
    top_3_preds = [option_letters[idx] for idx in ranked_indices[:3]]
    
    sample.loc[i, 'Prediction'] = " ".join(top_3_preds)

sample.to_csv('/kaggle/working/submission.csv', index=False)
print("Saved ensemble submission.csv successfully!")


Loading Tokenizer and Model from: /kaggle/input/models/tanmay240/electra-base-discriminator/pytorch/default/1/electra-base-discriminator


config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraModel LOAD REPORT from: google/electra-base-discriminator
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
electra.embeddings_project.weight                 | UNEXPECTED |  | 
electra.embeddings_project.bias                   | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Predicting FacebookAI/roberta-base: 100%|██████████| 500/500 [00:25<00:00, 19.96it/s]



Blending probabilities and ranking answers...
Saved ensemble submission.csv successfully!
